## 5 Idiom Classification

This notebook uses the decoder embeddings of the fine tuned whisper model to train an audio idiom classifier.

In [2]:
# Cell 1: Imports & Setup
import sys
import os
from pathlib import Path
import torch
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration

notebook_dir = Path.cwd()
whisper_dir = notebook_dir.parent
sys.path.append(str(whisper_dir))

from whisper_asr import load_all_data, extract_decoder_embeddings, train_classifier, extract_encoder_embeddings
from whisper_asr.utils import get_best_gpu

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device(f"cuda:{get_best_gpu()}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Selected GPU 7 with 24121 MiB free memory
Using device: cuda:7


In [3]:
# Cell 2: Load model & processor
MODEL_PATH = "../models/whisper-medium-rm-all-it"
processor = WhisperProcessor.from_pretrained(MODEL_PATH)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)
print("Model loaded.")

Loading weights: 100%|██████████| 947/947 [00:00<00:00, 1596.86it/s]


Model loaded.


In [4]:
# Cell 3: Load data (train and test splits)
train_df = load_all_data("train")
test_df = load_all_data("test")

In [6]:
# Cell 4: Extract embeddings for train and test sets
print("Extracting train encoder embeddings...")
train_embeddings = extract_encoder_embeddings(
    model, processor,
    train_df["audio_path"].tolist(),
    device=DEVICE, batch_size=8
)
print("Extracting test encoder embeddings...")
test_embeddings = extract_encoder_embeddings(
    model, processor,
    test_df["audio_path"].tolist(),
    device=DEVICE, batch_size=8
)

Extracting train encoder embeddings...


  2%|▏         | 62/3696 [01:02<1:00:45,  1.00s/it]


KeyboardInterrupt: 

In [ ]:
# Cell 5: Train classifier and evaluate
train_labels = train_df["idiom"].tolist()
test_labels = test_df["idiom"].tolist()

classifier, predictions = train_classifier(
    train_embeddings, train_labels,
    test_embeddings, test_labels
)

# Optionally save the classifier
joblib.dump(classifier, "../models/idiom_classifier.pkl")